In [ ]:
"""Pandas."""

## Библиотека Pandas

In [3]:
# Подключение к базе данных SQL
import sqlite3 as sql

import numpy as np
import pandas as pd
import requests
from pandas import Index

### Объекты DataFrame и Series

#### Создание датафрейма

In [ ]:
# из архива .zip, который содержит только один файл
csv_zip = pd.read_csv("./content/train.zip")
csv_zip.head(3)

In [ ]:
# из excel-файла
excel_data = pd.read_excel("./content/iris.xlsx", sheet_name=0)
excel_data.head(3)

In [ ]:
# из элементов (таблицы) на web-странице
# используем requests с правильными заголовками для обхода блокировки
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(
    "https://en.wikipedia.org/wiki/World_population", headers=headers
)

html_data = pd.read_html(response.text, match="World population")

In [ ]:
len(html_data)

In [ ]:
html_data[0]

In [ ]:
# подключение
conn = sql.connect("./content/chinook.db")

# выберем все строки из таблицы tracks
sql_data = pd.read_sql("SELECT * FROM tracks;", conn)

sql_data.head(3)

#### Создание датафрейма из словаря

In [ ]:
# создадим несколько списков и массивов Numpy
country = np.array(
    [
        "China",
        "Vietnam",
        "United Kingdom",
        "Russia",
        "Argentina",
        "Bolivia",
        "South Africa",
    ]
)
capital = np.array(
    ["Beijing", "Hanoi", "London", "Moscow", "Buenos Aires", "Sucre", "Pretoria"]
)
population = np.array([1400, 97, 67, 144, 45, 12, 59])  # млн. человек
area = np.array([9.6, 0.3, 0.2, 17.1, 2.8, 1.1, 1.2])  # млн. кв. км.
sea = np.array([1] * 5 + [0, 1])

In [ ]:
countries_dict = {}

countries_dict["country"] = country
countries_dict["capital"] = capital
countries_dict["population"] = population
countries_dict["area"] = area
countries_dict["sea"] = sea

countries_dict

In [ ]:
# создадим датафрейм
countries = pd.DataFrame(countries_dict)
countries

#### Создание датафрейма из 2D массива Numpy

In [ ]:
arr = np.array([[1, 1, 1], [2, 2, 2], [3, 3, 3]])
pd.DataFrame(arr)

### Структура и свойства датафрейма

In [ ]:
# названия столбцов
countries.columns

In [ ]:
# способ идентификации строк
countries.index

In [ ]:
# значения
countries.values

In [ ]:
# выведем описание индекса датафрейма через атрибус axes[0]
countries.axes[0]

In [ ]:
# axes[1] выводит названия столбцов
countries.axes[1]

In [ ]:
# количество измерений
print(countries.ndim)
# размерность
print(countries.shape)
# общее кол-во элементов
print(countries.size)

In [ ]:
# типы каждого столбца
countries.dtypes

In [ ]:
# объем занимаемой памяти по столбцам
countries.memory_usage()

### Индекс

#### Присвоение индекса

In [ ]:
# в датафейме можно задать собственный индекс
# например коды стран
custom_index: "Index[str]" = pd.Index(
    ["CN", "VN", "GB", "RU", "AR", "BO", "ZA"], dtype="string"
)

In [ ]:
countries = pd.DataFrame(countries_dict, index=custom_index)
countries

In [ ]:
# сброс индекса
# параметр inplace = True сохраняет изменения
countries.reset_index(inplace=True)
countries

In [ ]:
# прошлый индекс стал отдельным столбцом
# чтобы снова сделать индексом .set_index
countries.set_index("index", inplace=True)
countries

In [ ]:
# снова сбросим но без сохранения
countries.reset_index(drop=True, inplace=True)
countries

In [ ]:
# собственный индекс
countries.index = custom_index
countries

#### Многоуровневые индекс и названия столбцов

In [ ]:
multiple_rows = [
    ("Asia", "CN"),
    ("Asia", "VN"),
    ("Europe", "GB"),
    ("Europe", "RU"),
    ("S. America", "AR"),
    ("S. America", "BO"),
    ("Africa", "ZA"),
]

multiple_cols = [
    ("names", "country"),
    ("names", "capital"),
    ("data", "population"),
    ("data", "area"),
    ("data", "sea"),
]

# мульти-индексы из кортежей
custom_multindex = pd.MultiIndex.from_tuples(multiple_rows, names=["region", "code"])

# мульти-столбцы из кортежей
custom_multicols = pd.MultiIndex.from_tuples(multiple_cols)

# передадим в датафрейм
countries.index = custom_multindex
countries.columns = custom_multicols

countries

In [ ]:
# вернемся к обычному индексу и названиям столбцов
custom_cols1: "Index[str]" = pd.Index(
    ["country", "capital", "population", "area", "sea"], dtype="string"
)

In [ ]:
countries.index = custom_index

In [ ]:
countries.columns = custom_cols1

In [ ]:
print(countries)

### Преобразование в другие форматы

In [ ]:
# в словарь
print(countries.to_dict())

In [ ]:
# Numpy=array
countries.to_numpy()

In [ ]:
# файл
# index = False, чтобы не сохранять индекс
countries.to_csv("./content/countries.csv", index=False)

In [ ]:
# Series в список
print(countries.country.to_list())

### Создание Series

#### Создание Series из списка

In [ ]:
country_list = [
    "China",
    "South Africa",
    "United Kingdom",
    "Russia",
    "Argentina",
    "Vietnam",
    "Australia",
]

In [ ]:
country_series_1: pd.Series[str] = pd.Series(country_list)
country_series_1

In [ ]:
# доступ к элементам по индексу
country_series_1[0]

#### Создание Series из словаря

In [ ]:
country_dict = {
    "CN": "China",
    "ZA": "South Africa",
    "GB": "United Kingdom",
    "RU": "Russia",
    "AR": "Argentina",
    "VN": "Vietnam",
    "AU": "Australia",
}

In [ ]:
country_series = pd.Series(country_dict)
country_series

CN             China
ZA      South Africa
GB    United Kingdom
RU            Russia
AR         Argentina
VN           Vietnam
AU         Australia
dtype: str

In [ ]:
# теперь для доступа к элементам можно использовать индекс
# которым у нас являются коды стран
country_series["AU"]

## Доступ к строкам и столбцам

### Циклы в датафрейме

In [ ]:
for columns in countries:
    print(columns)

In [ ]:
# .iterrows() - Series с индексом каждой строки и её значением
for index, row in countries.iterrows():
    print(index)
    print(row)
    print("...")
    print(type(row))
    break

In [ ]:
# использование данных определённой строки
for _, row in countries.iterrows():
    print(row["capital"] + " is the capital of " + row["country"])
    break

### Доступ к столбцам

In [ ]:
# в отличие от Series, в датафрейме через квадратные скобки
# происходит доступ к столбцам

countries["capital"]

In [ ]:
# также можно через точку, но без пробелов

countries.capital

In [ ]:
# отдельные столбцы - Series

print(type(countries.capital))
print(type(countries["capital"]))

In [ ]:
# внутренние скобки - список столбцов
# внешние скобки - сам оператор индексации
# поэтому на выходи получится DataFrame, а не Series
countries[["capital"]]

In [ ]:
# к нескольким столбцам
countries[["capital", "area"]]

In [ ]:
# через метод .filter()
# с параметром items
countries.filter(items=["capital", "population"])

### Доступ к строкам

In [ ]:
# доступ к строкам с помощью индекса
# не включая верхнюю границу
countries[1:5]

### Методы `.loc[]` и `iloc[]`

#### Метод `.loc[]`

In [ ]:
# label-based location
# метод .loc[] позволяет получить доступ
# к строкам и столбцам по их названиям
countries.loc[["CN", "RU", "VN"], ["capital", "population", "area"]]

In [ ]:
# через : можно вывести все столбцы/строки
countries.loc[:, ["capital", "population", "area"]]

In [ ]:
# также можно передавать значения Boolean
# чтобы фильтровать данные по условию
countries.loc[:, [False, False, False, False, True]]

#### Метод `.get_loc()`

In [ ]:
# атрибут index и метод .get_loc
# позволяют вывести порядковый номер
# строки по индексу (начиная с нуля)
countries.index.get_loc("RU")

In [ ]:
# атрибут columns и метод .get_loc
# позволяют вывести порядковый номер
# столбца по названию (начиная с нуля)
countries.columns.get_loc("country")

Метод `.iloc[]`

In [ ]:
# integer-based location
# метод .iloc[] позволяет получить доступ
# к строкам и столбцам по числовому индексу
countries.iloc[[0, 3, 5], [0, 1, 2]]

In [ ]:
# можно использовать срезы
countries.iloc[:3, -2:]

In [ ]:
# удобно использовать доступ
# с помощью квадратных скобок
# и метода .iloc[]
countries[["population", "area"]].iloc[[0, 3]]

#### Доступ по многоуровневому индексу

In [ ]:
countries.index = custom_multindex
countries.columns = custom_multicols

countries

In [ ]:
# доступ к первой строке
# с помощью двойного индекса
# и метода .loc[]
countries.loc["Asia", "CN"]

In [ ]:
# также можно передавать значения в
# форме кортежей для строк и столбцов
countries.loc[
    ("Asia", "CN"),  # мульти-индексы
    [
        ("data", "population"),  # мульти-названия столбцов
        ("data", "area"),
        ("data", "sea"),
    ],
]

In [ ]:
# доступ к строкам можно получить,
# указав внутри кортежа название региона,
# список с кодами стран
countries.loc[("Asia", ["CN", "VN"]), :]

In [ ]:
# можно указать только регион
# тогда мы получим все страны,
# которые в него входят
countries.loc[("Asia"), :]

In [ ]:
# аналогично можно получить доступ к столбцам
countries.loc[:, [("names", "country"), ("data", "population")]]

In [ ]:
# метод .iloc[] игнорирует структуру многоуровневого индекса
# и использует простой числовой индекс
countries.iloc[3, [2, 3, 4]]

### Метод `.xs()`

In [ ]:
# cross-secton
# позволяет получить доступ к определённому уровню
# многоуровневого индекса
# на уровне 'r
# axis = 0, чтобы отбирались строки
countries.xs("Europe", level="region", axis=0)

In [ ]:
# на первом уровне выберем 'names'
# на втором уровне выберем 'country'
# axis = 1, чтобы отбирались столбцы
countries.xs(("names", "country"), level=(0, 1), axis=1)

In [ ]:
# можно разбить на два xs
# чтобы не использовать level
countries.xs("names", axis=1, level=0).xs("Europe", axis=0)

In [ ]:
# вернём одноуровневость
countries.index = custom_index
countries.columns = custom_cols1

countries

### Метод `.at[]`

In [ ]:
# только для одной ячейки
countries.at["CN", "capital"]

### Фильтры

#### Логическая маска

In [ ]:
# создадим логическую маску
countries.population > 1000

In [ ]:
# применим логическую маску
countries[countries.population > 1000]

In [ ]:
# & - логическое И
countries[(countries.population > 50) & (countries.area < 2)]

In [ ]:
# | - логическое ИЛИ
population_mask = countries.population > 70
area_mask = countries.population < 50

mask = population_mask | area_mask
countries[mask]

### Метод `.query()` 

In [ ]:
# условия дословно
countries.query("population > 50 and area < 2")

In [ ]:
# использование кавычек
countries.query("country == 'United Kingdom'")

### Другие способы фильтрации

In [ ]:
# проверка вхождения
keyword_list = ["Beijing", "Moscow", "Hanoi"]

print(countries[countries.capital.isin(keyword_list)])

In [ ]:
# строка начинается с ...
# ~ - логическое НЕ
print(countries[~countries.country.str.startswith("A")])

In [ ]:
# n наибольших
countries.nlargest(3, "population")

In [ ]:
# n наименьших
countries.nsmallest(3, "population")

In [ ]:
# argmax() индекс наибольшей
# argmin() индекс наименьшей
countries.area.argmax()

In [ ]:
# соответствующая страна
print(countries.iloc[[int(countries.area.argmax())]])

In [ ]:
# логические маски можно использовать с loc
countries.loc[countries.population > 90, :]

In [ ]:
# .filter с параметром like позволяет искать совпадения в
# индексе (axis = 0) или столбцах (axis = 1)
countries.filter(like="ZA", axis=0)

### Сортировка

In [ ]:
countries.sort_values(by="population", inplace=False, ascending=True)  # восходящий

In [ ]:
countries.sort_values(
    by=["area", "population"], inplace=False, ascending=False  # нисходящий
)

In [ ]:
# сортировка по индексу
countries.sort_index()